## 09 Population

**笔记本**：`09 population20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`rasterio`（含 `mask`、`rasterize`）、`matplotlib`、`shapely`、`requests` 等。

**输入**：
- `../04 Transport/data_raw/shenzhen_boundary.geojson`
- `../06 Buildings/data_out/sz_buildings.gpkg`、`../08 POI Demand/data_out/sz_demand_grid.gpkg`（`h3_id` 与 `../03 Boundary/data_out/sz_hex_grid_res8.gpkg` 对齐）
- WorldPop：`../00/13-population-density-worldpop/guangdong_chn_ppp_2020_unadj.tif`（优先），否则 `data_raw/chn_ppp_2020_constrained_100m.tif` 或联网下载全国 tif
- `../00/11-public-facilities/SHP/shenzhen_residential_compound_points.shp`（小区点，可选）
- 百度：data_raw/baidu_heatmap_cache.json（无则调 API 并写入缓存）

**做了什么 / 算了什么**：
- 将 WorldPop 裁剪为 data_out/sz_worldpop_clipped.tif。
- 人口为 zonal sum（rasterize + bincount，all_touched=False）。
- 住宅：usage_cat == residential 建筑质心落入格，汇总栋数、体积、均高等。
人口栅格告诉你“人在哪”， 住宅建筑告诉你“承载居住的人居形态在哪”。
- 小区点：落入格的 xiaoqu_count。
因为“有人口”不一定等于“有典型社区配送场景”。而小区点数量多，通常更直接意味着：社区配送任务多；住宅末端进入场景明确；更像外卖/快递/生鲜的典型落点
- 百度 POI 活动：写入格级 weekend_hotspot_*。
- 合并 08：demand_pressure、employment_proxy、commercial_activity（最近邻对齐）。
把纯人口层推进成：人口 + 居住 + 商业 + 活动 + 工作 的综合强度层
- intensity_index：pop_norm、res_norm、demand_norm 加权 0.4 / 0.3 / 0.3。
某个 H3 格是否既有人口基础，又有住宅承载，又有服务/活动需求，从而更可能形成稳定且有意义的配送场景。

**写出文件**：
- `data_out/sz_worldpop_clipped.tif`
- `data_out/sz_population_grid.gpkg`

**典型输出信息**：栅格尺寸、有效像元、网格人口合计、最大密度、格网保存路径。


In [1]:
# ============================================================
#  09 Population / Residential Intensity / Activity Proxy
#  数据源: WorldPop 100m + 06 Buildings + 08 POI Demand
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
from shapely.geometry import box, mapping
from shapely.prepared import prep as shapely_prep
import requests
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
BUILDINGS_06 = Path("../06 Buildings/data_out/sz_buildings.gpkg")
DEMAND_08 = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")

# 优先用 00 已有的广东省 WorldPop (不需要下载全国 tif)
WORLDPOP_LOCAL = Path("../00/13-population-density-worldpop/guangdong_chn_ppp_2020_unadj.tif")
WORLDPOP_TIF = WORLDPOP_LOCAL if WORLDPOP_LOCAL.exists() else RAW / "chn_ppp_2020_constrained_100m.tif"
CLIPPED_TIF = OUT / "sz_worldpop_clipped.tif"

# 00 小区点 (补充住宅密度)
XIAOQU_PT = Path("../00/11-public-facilities/SHP/shenzhen_residential_compound_points.shp")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"06 buildings: {BUILDINGS_06.exists()}")
print(f"08 demand grid: {DEMAND_08.exists()}")
print(f"WorldPop tif: {WORLDPOP_TIF} -> exists: {WORLDPOP_TIF.exists()}")
print(f"小区点: {XIAOQU_PT.exists()}")

06 buildings: True
08 demand grid: True
WorldPop tif: ../00/13-population-density-worldpop/guangdong_chn_ppp_2020_unadj.tif -> exists: True
小区点: True


/Users/shirly/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ============================================================
#  1. WorldPop 人口栅格
#  优先用 00/ 已有的广东省裁剪版, 无需下载
#  如果 00 里没有, 才下载全国版
# ============================================================

if WORLDPOP_TIF.exists():
    size_mb = WORLDPOP_TIF.stat().st_size / 1e6
    print(f"Using local WorldPop: {WORLDPOP_TIF} ({size_mb:.0f} MB)")
else:
    WORLDPOP_URL = "https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/CHN/chn_ppp_2020_constrained.tif"
    print(f"Local not found. Downloading from {WORLDPOP_URL} ...")
    r = requests.get(WORLDPOP_URL, stream=True, timeout=30)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    dl_path = RAW / "chn_ppp_2020_constrained_100m.tif"
    with open(dl_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                print(f"\r  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB", end="", flush=True)
    WORLDPOP_TIF = dl_path
    print(f"\n  Saved -> {WORLDPOP_TIF}")

Using local WorldPop: ../00/13-population-density-worldpop/guangdong_chn_ppp_2020_unadj.tif (1 MB)


In [3]:
# ============================================================
#  2. 裁剪栅格到深圳 + H3 res=8 分析网格（与 08 demand 对齐）
#  WorldPop 像元保持原分辨率；人口按「格内像元求和」(zonal sum)
# ============================================================
from rasterio.features import rasterize

# ── 裁剪 WorldPop 到深圳 ──
if not CLIPPED_TIF.exists():
    print("Clipping WorldPop to Shenzhen ...")
    with rasterio.open(WORLDPOP_TIF) as src:
        geom = [mapping(shenzhen_geom)]
        out_image, out_transform = rio_mask(src, geom, crop=True, nodata=-99999)
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": -99999,
        })
        with rasterio.open(CLIPPED_TIF, "w", **out_meta) as dst:
            dst.write(out_image)
    print(f"  Saved -> {CLIPPED_TIF}")
else:
    print(f"Clipped raster exists: {CLIPPED_TIF}")

# ── 读取裁剪后栅格 ──
with rasterio.open(CLIPPED_TIF) as src:
    pop_data = src.read(1)
    pop_transform = src.transform
    pop_crs = src.crs
    src_nodata = src.nodata
    print(f"  Shape: {pop_data.shape}, CRS: {pop_crs}")
    print(f"  Valid cells: {(pop_data > 0).sum():,}")
    print(f"  Total population: {pop_data[pop_data > 0].sum():,.0f}")

# ── 分析网格：与 08 demand 相同 H3 单元 ──
if not DEMAND_08.exists():
    raise FileNotFoundError(
        f"需要先有 08 的 H3 格网: {DEMAND_08}\n请先运行 08 poi_demand20260403.1.ipynb"
    )

pop_grid = gpd.read_file(DEMAND_08)[["h3_id", "geometry"]].copy()
pop_grid["cx"] = pop_grid.geometry.centroid.x
pop_grid["cy"] = pop_grid.geometry.centroid.y
pop_grid["_zix"] = np.arange(len(pop_grid), dtype=np.int32)

# 六边形面积（km²）用于密度
_gm = pop_grid.to_crs(32650)
pop_grid["_area_km2"] = _gm.geometry.area / 1e6

# ── Zonal sum ──
nd = src_nodata if src_nodata is not None else -99999
pop_clean = np.where(np.isfinite(pop_data) & (pop_data != nd), np.maximum(pop_data, 0), 0.0)

geom_zix = [(mapping(geom), int(z)) for geom, z in zip(pop_grid.geometry, pop_grid["_zix"])]
rid = rasterize(
    geom_zix,
    out_shape=pop_data.shape,
    transform=pop_transform,
    fill=-1,
    dtype=np.int32,
    all_touched=False,
)

flat_rid = rid.ravel()
flat_pop = pop_clean.ravel()
inside = flat_rid >= 0
n_bins = len(pop_grid)
sums = np.bincount(flat_rid[inside], weights=flat_pop[inside], minlength=n_bins)
pop_grid["pop_count"] = sums[pop_grid["_zix"].to_numpy()]
pop_grid["pop_density"] = np.where(
    pop_grid["_area_km2"] > 0, pop_grid["pop_count"] / pop_grid["_area_km2"], 0.0
)

print(f"\nGrid: {len(pop_grid):,} H3 cells")
print(f"Cells with population > 0: {(pop_grid['pop_count'] > 0).sum():,}")
print(f"Total grid population (zonal sum): {pop_grid['pop_count'].sum():,.0f}")
print(f"Max density: {pop_grid['pop_density'].max():,.0f} /km²")
unassigned = flat_pop[flat_rid < 0].sum()
print(f"  (valid pixels not in any grid cell: {unassigned:,.0f} pop)")


Clipped raster exists: data_out/sz_worldpop_clipped.tif
  Shape: (551, 1052), CRS: EPSG:4326
  Valid cells: 120,461
  Total population: 15,240,720

Grid: 2,754 H3 cells
Cells with population > 0: 2,156
Total grid population (zonal sum): 15,134,766
Max density: 42,053 /km²
  (valid pixels not in any grid cell: 105,954 pop)


In [4]:
# ============================================================
#  3. 住宅强度 (from 06 Buildings)
#  读取已有建筑数据, 按网格汇总住宅建筑体积和栋数
# ============================================================

if BUILDINGS_06.exists():
    print("Loading buildings ...")
    bldg = gpd.read_file(BUILDINGS_06)

    # 只取住宅
    res = bldg[bldg["usage_cat"] == "residential"].copy()
    print(f"  Residential buildings: {len(res):,}")

    # 计算体积
    res["volume_m3"] = res["footprint_m2"] * res["height_m"]

    # 质心 → 空间连接到网格
    res_pts = res.copy()
    res_pts["geometry"] = res_pts.geometry.centroid
    joined = gpd.sjoin(res_pts, pop_grid[["h3_id", "geometry"]], how="inner", predicate="within")

    res_stats = joined.groupby("h3_id").agg(
        residential_count=("height_m", "count"),
        residential_volume=("volume_m3", "sum"),
        residential_avg_height=("height_m", "mean"),
    ).reset_index()

    pop_grid = pop_grid.merge(res_stats, on="h3_id", how="left")
    pop_grid[["residential_count", "residential_volume", "residential_avg_height"]] = \
        pop_grid[["residential_count", "residential_volume", "residential_avg_height"]].fillna(0)

    del bldg, res, res_pts, joined
    print(f"  Grids with residential buildings: {(pop_grid['residential_count'] > 0).sum():,}")
else:
    print("06 Buildings not found, skipping residential intensity")
    pop_grid["residential_count"] = 0
    pop_grid["residential_volume"] = 0
    pop_grid["residential_avg_height"] = 0

# ── 补充: 00 小区点 → 小区密度 ──
if XIAOQU_PT.exists():
    print("\nLoading 小区点 ...")
    xq_pts = gpd.read_file(XIAOQU_PT).to_crs(4326)
    xq_pts = gpd.clip(xq_pts, shenzhen_geom)
    print(f"  小区点: {len(xq_pts):,}")

    xq_joined = gpd.sjoin(xq_pts, pop_grid[["h3_id", "geometry"]], how="inner", predicate="within")
    xq_stats = xq_joined.groupby("h3_id").size().reset_index(name="xiaoqu_count")
    pop_grid = pop_grid.merge(xq_stats, on="h3_id", how="left")
    pop_grid["xiaoqu_count"] = pop_grid["xiaoqu_count"].fillna(0).astype(int)
    del xq_pts, xq_joined
    print(f"  Grids with 小区: {(pop_grid['xiaoqu_count'] > 0).sum():,}")
else:
    pop_grid["xiaoqu_count"] = 0
    print("小区点 not found, skipping")

Loading buildings ...
  Residential buildings: 402,059
  Grids with residential buildings: 2,105

Loading 小区点 ...
  小区点: 12,125
  Grids with 小区: 1,453


In [5]:
# ============================================================
#  3b. 百度热力图 — 周末/节假日活动热点 proxy
#  用百度地图 热力图 API 抓取人群密度快照
#  API: https://lbsyun.baidu.com/faq/api?title=webapi-heat
#  注: 百度热力图 API 返回的是实时快照, 建议周末下午抓一次
# ============================================================

BAIDU_AK = "WBjHYGJcg9il6cUXt5YzFwsFqGc63Er7"
HEATMAP_CACHE = RAW / "baidu_heatmap_cache.json"

import json, time

def fetch_baidu_heatmap(city="深圳", ak=BAIDU_AK):
    """抓取百度地图城市热力图数据 (实时人群密度)"""
    url = "https://huiyan.baidu.com/migration/cityrank.jsonp"
    params = {
        "dt": "city",
        "id": "440300",
        "type": "move",
        "date": time.strftime("%Y%m%d"),
        "callback": "",
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        text = r.text.strip()
        if text.startswith("("):
            text = text[1:-1]
        return json.loads(text)
    except Exception as e:
        print(f"  Baidu heatmap error: {e}")
        return None

# 百度热力图的公开 API 有限制, 替代方案:
# 用百度 Place API 搜索景区/商圈/公园的评论数/评分作为活动 proxy
def fetch_baidu_poi_activity(query, region="深圳", ak=BAIDU_AK, tag="旅游景点"):
    """用百度地点搜索 API 获取 POI 及评论数作为活动 proxy"""
    url = "https://api.map.baidu.com/place/v2/search"
    all_pois = []
    page = 0
    while True:
        params = {
            "query": query,
            "region": region,
            "output": "json",
            "ak": ak,
            "page_size": 20,
            "page_num": page,
            "scope": 2,
        }
        try:
            r = requests.get(url, params=params, timeout=15)
            data = r.json()
            if data.get("status") != 0:
                break
            results = data.get("results", [])
            if not results:
                break
            for p in results:
                loc = p.get("location", {})
                detail = p.get("detail_info", {})
                all_pois.append({
                    "name": p.get("name", ""),
                    "lon": loc.get("lng", 0),
                    "lat": loc.get("lat", 0),
                    "overall_rating": float(detail.get("overall_rating", 0)),
                    "comment_num": int(detail.get("comment_num", 0)),
                    "tag": tag,
                })
            page += 1
            time.sleep(0.3)
            if page >= 20:
                break
        except Exception as e:
            print(f"  Error: {e}")
            break
    return all_pois

# 抓取深圳热门场景的活动数据
if HEATMAP_CACHE.exists():
    print(f"Loading cache: {HEATMAP_CACHE}")
    with open(HEATMAP_CACHE) as f:
        activity_pois = json.load(f)
else:
    print("Fetching Baidu POI activity data ...")
    activity_pois = []
    for query, tag in [("公园", "park"), ("景区", "scenic"), ("商圈", "commercial"),
                        ("购物中心", "mall"), ("步行街", "street")]:
        pois = fetch_baidu_poi_activity(query, tag=tag)
        activity_pois.extend(pois)
        print(f"  {tag}: {len(pois)} POIs")
        time.sleep(0.5)

    with open(HEATMAP_CACHE, "w") as f:
        json.dump(activity_pois, f, ensure_ascii=False)
    print(f"Total: {len(activity_pois)} activity POIs, cached.")

# 转 GeoDataFrame
if activity_pois:
    act_df = pd.DataFrame(activity_pois)
    act_gdf = gpd.GeoDataFrame(
        act_df, geometry=gpd.points_from_xy(act_df["lon"], act_df["lat"]), crs=4326
    )
    act_gdf = gpd.clip(act_gdf, shenzhen_geom)
    # 注: 百度坐标 BD09 → WGS84 有偏移, 此处近似使用
    print(f"  Activity POIs in Shenzhen: {len(act_gdf)}")

    # 空间连接到 pop_grid
    act_joined = gpd.sjoin(act_gdf, pop_grid[["h3_id", "geometry"]], how="inner", predicate="within")
    act_stats = act_joined.groupby("h3_id").agg(
        weekend_hotspot_count=("comment_num", "count"),
        weekend_hotspot_score=("comment_num", "sum"),
    ).reset_index()
    pop_grid = pop_grid.merge(act_stats, on="h3_id", how="left")
    pop_grid[["weekend_hotspot_count", "weekend_hotspot_score"]] = \
        pop_grid[["weekend_hotspot_count", "weekend_hotspot_score"]].fillna(0)
    print(f"  Grids with hotspots: {(pop_grid['weekend_hotspot_count'] > 0).sum()}")
else:
    pop_grid["weekend_hotspot_count"] = 0
    pop_grid["weekend_hotspot_score"] = 0
    print("  No activity data available")

Loading cache: data_raw/baidu_heatmap_cache.json
  Activity POIs in Shenzhen: 466
  Grids with hotspots: 311


In [6]:
# ============================================================
#  4. 合并 08 活动强度 + 合成 intensity_index + 保存
# ============================================================

# ── 合并 08 demand_pressure（同 h3_id 直接对齐）──
if DEMAND_08.exists():
    print("Loading 08 demand grid ...")
    demand = gpd.read_file(DEMAND_08)
    dcols = ["h3_id"]
    if "demand_pressure" in demand.columns:
        dcols.append("demand_pressure")
    elif "poi_total" in demand.columns:
        dcols.append("poi_total")
    for c in ("office_count", "food_count", "retail_count"):
        if c in demand.columns:
            dcols.append(c)
    sub = demand[dcols].copy()
    if "poi_total" in sub.columns and "demand_pressure" not in sub.columns:
        sub = sub.rename(columns={"poi_total": "demand_pressure"})
    pop_grid = pop_grid.merge(sub, on="h3_id", how="left")
    pop_grid["demand_pressure"] = pop_grid["demand_pressure"].fillna(0)
    if "office_count" in pop_grid.columns:
        pop_grid["employment_proxy"] = pop_grid["office_count"].fillna(0)
        print("  Merged employment_proxy")
    else:
        pop_grid["employment_proxy"] = 0
    fc, rc = "food_count" in pop_grid.columns, "retail_count" in pop_grid.columns
    if fc or rc:
        parts = []
        if fc:
            parts.append(pop_grid["food_count"].fillna(0))
        if rc:
            parts.append(pop_grid["retail_count"].fillna(0))
        pop_grid["commercial_activity"] = sum(parts)
        print("  Merged commercial_activity")
    else:
        pop_grid["commercial_activity"] = 0
    print("  Merged demand_pressure")
else:
    print("08 demand grid not found, skipping")
    pop_grid["demand_pressure"] = 0

# ── 合成 intensity_index ──
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

pop_grid["pop_norm"] = norm(pop_grid["pop_count"])
pop_grid["res_norm"] = norm(pop_grid["residential_volume"])
pop_grid["demand_norm"] = norm(pop_grid["demand_pressure"])

W_POP = 0.40
W_RES = 0.30
W_DEMAND = 0.30

pop_grid["intensity_index"] = (
    W_POP * pop_grid["pop_norm"]
    + W_RES * pop_grid["res_norm"]
    + W_DEMAND * pop_grid["demand_norm"]
)

# ── 保存 ──
out_cols = [
    "h3_id",
    "pop_count",
    "pop_density",
    "residential_count",
    "residential_volume",
    "residential_avg_height",
    "demand_pressure",
    "employment_proxy",
    "commercial_activity",
    "intensity_index",
    "geometry",
]
for c in [
    "xiaoqu_count",
    "weekend_hotspot_count",
    "weekend_hotspot_score",
]:
    if c in pop_grid.columns:
        out_cols.append(c)

result = pop_grid[[c for c in out_cols if c in pop_grid.columns]].copy()
# 去掉辅助列若误入
for aux in ["_zix", "_area_km2", "cx_r", "cy_r"]:
    if aux in result.columns:
        result = result.drop(columns=[aux])

result.to_file(OUT / "sz_population_grid.gpkg", driver="GPKG")
print(f"\\nSaved -> data_out/sz_population_grid.gpkg  ({len(result):,} cells)")

# ── 统计摘要 ──
g = result[result["pop_count"] > 0]
print(f"\\n=== Summary (cells with pop > 0: {len(g):,}) ===")
print(f"Population:   total={g['pop_count'].sum():,.0f}  mean={g['pop_count'].mean():.0f}  max={g['pop_count'].max():.0f}")
print(f"Pop density:  mean={g['pop_density'].mean():,.0f} /km²  max={g['pop_density'].max():,.0f} /km²")
print(f"Res volume:   mean={g['residential_volume'].mean():,.0f} m³  max={g['residential_volume'].max():,.0f} m³")
print(f"Intensity:    mean={g['intensity_index'].mean():.3f}  max={g['intensity_index'].max():.3f}")

print("\\n=== Top 10 Highest Intensity (h3_id) ===")
top10 = result.nlargest(10, "intensity_index")[
    ["h3_id", "pop_count", "residential_count", "demand_pressure", "intensity_index"]
]
print(top10.to_string(index=False))


Loading 08 demand grid ...
  Merged employment_proxy
  Merged commercial_activity
  Merged demand_pressure
\nSaved -> data_out/sz_population_grid.gpkg  (2,754 cells)
\n=== Summary (cells with pop > 0: 2,156) ===
Population:   total=15,134,766  mean=7020  max=30225
Pop density:  mean=9,774 /km²  max=42,053 /km²
Res volume:   mean=13,907,428 m³  max=396,358,901 m³
Intensity:    mean=0.128  max=0.709
\n=== Top 10 Highest Intensity (h3_id) ===
          h3_id    pop_count  residential_count  demand_pressure  intensity_index
88411caa1bfffff 20005.792542              962.0           273.40         0.708749
88411cb9a1fffff 28317.470428              564.0           316.35         0.681699
88411cb9b5fffff 22035.420418              267.0           117.10         0.676067
88411cb9b3fffff 28501.654236              112.0           305.20         0.654088
88411cb9a3fffff 28358.938263              186.0           314.00         0.627218
88411caa0dfffff 30224.816833               15.0           259.95